In [ ]:
!pip install 'lerobot[pi0]' av -q
!pip install 'lerobot[dataset]'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
'''import shutil

src = "/content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2/checkpoints/004000"
dst = "/content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2_step4000_backup"

shutil.copytree(src, dst)
print("Backup done")'''

Backup done


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import sys
sys.argv = ["lerobot-train"]
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Fix vision tower key mismatch in Pi0.5 loading code
path = "/usr/local/lib/python3.12/dist-packages/lerobot/policies/pi05/modeling_pi05.py"
src = open(path).read()
if 'vision_tower.vision_model.' not in src or 'new_key.replace("vision_tower.vision_model."' not in src:
    target = '            # Handle vision tower embedding layer potential differences'
    fix = '            if "vision_tower.vision_model." in new_key:\n                new_key = new_key.replace("vision_tower.vision_model.", "vision_tower.")\n\n'
    open(path, 'w').write(src.replace(target, fix + target))
    print("Vision tower key fix applied")
else:
    print("Already patched")

from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.scripts.lerobot_train import train
from lerobot.configs import PreTrainedConfig, FeatureType, PolicyFeature
from pathlib import Path

BASE_MODEL = "lerobot/pi05_base"
policy_cfg = PreTrainedConfig.from_pretrained(BASE_MODEL)
policy_cfg.pretrained_path = BASE_MODEL
policy_cfg.repo_id = "subhodipsaha/pi05_stack_three_cubes_08_19_v2"
policy_cfg.input_features = {
    "observation.state": PolicyFeature(type=FeatureType.STATE, shape=(6,)),
    "observation.images.phone": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 224, 224)),
    "observation.images.wrist": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 224, 224)),
}
policy_cfg.output_features = {
    "action": PolicyFeature(type=FeatureType.ACTION, shape=(6,)),
}
policy_cfg.train_expert_only = True
policy_cfg.gradient_checkpointing = True
policy_cfg.dtype = "bfloat16"
policy_cfg.use_amp = True

cfg = TrainPipelineConfig(
    dataset=DatasetConfig(
        repo_id="subhodipsaha/so101_stack_three_cubes_08_19",
        video_backend="pyav",
    ),
    policy=policy_cfg,
    output_dir=Path("/content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2"),
    steps=5000,
    batch_size=16,
    num_workers=0,
    log_freq=100,
    save_freq=1000,
)
train(cfg)

Vision tower key fix applied
[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


config.json:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

INFO 2026-08-21 16:21:50 ot_train.py:272 {'batch_size': 16,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataloader_multiprocessing_context': 'spawn',
 'dataset': {'depth_output_unit': 'mm',
             'episodes': None,
             'eval_split': 0.0,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

policy_postprocessor.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

INFO 2026-08-21 16:25:29 ot_train.py:402 Creating optimizer and scheduler
INFO 2026-08-21 16:25:29 hedulers.py:155 Auto-scaling LR scheduler: num_training_steps (5000) < num_decay_steps (30000). Scaling warmup: 1000 → 166, decay: 30000 → 5000 (scale factor: 0.167)
INFO 2026-08-21 16:25:29 ot_train.py:434 Output dir: /content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2
INFO 2026-08-21 16:25:29 ot_train.py:441 cfg.steps=5000 (5K)
INFO 2026-08-21 16:25:29 ot_train.py:442 dataset.num_frames=29148 (29K)
INFO 2026-08-21 16:25:29 ot_train.py:443 dataset.num_episodes=39
INFO 2026-08-21 16:25:29 ot_train.py:446 Effective batch size: 16 x 1 = 16
INFO 2026-08-21 16:25:29 ot_train.py:447 num_learnable_params=693422112 (693M)
INFO 2026-08-21 16:25:29 ot_train.py:448 num_total_params=4143404816 (4B)
Training:  20%|██        | 1000/5000 [1:46:53<7:08:15,  6.42s/step]INFO 2026-08-21 18:12:23 ot_train.py:641 step:1K smpl:16K ep:21 epch:0.55 loss:0.082 grdn:0.984 lr:2.3e-05 updt_s:6.134 data_s:0.280 s

In [ ]:
import sys
sys.argv = ["lerobot-train"]
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

path = "/usr/local/lib/python3.13/dist-packages/lerobot/policies/pi05/modeling_pi05.py"
src = open(path).read()
if 'new_key.replace("vision_tower.vision_model."' not in src:
    target = '            # Handle vision tower embedding layer potential differences'
    fix = '            if "vision_tower.vision_model." in new_key:\n                new_key = new_key.replace("vision_tower.vision_model.", "vision_tower.")\n\n'
    open(path, 'w').write(src.replace(target, fix + target))
    print("Vision tower key fix applied")
else:
    print("Already patched")

from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.scripts.lerobot_train import train
from lerobot.configs import PreTrainedConfig, FeatureType, PolicyFeature
from pathlib import Path

CHECKPOINT = "/content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2/checkpoints/004000/pretrained_model"

policy_cfg = PreTrainedConfig.from_pretrained(CHECKPOINT)
policy_cfg.pretrained_path = CHECKPOINT
policy_cfg.repo_id = "subhodipsaha/pi05_stack_three_cubes_08_19_v2"
policy_cfg.input_features = {
    "observation.state": PolicyFeature(type=FeatureType.STATE, shape=(6,)),
    "observation.images.phone": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 224, 224)),
    "observation.images.wrist": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 224, 224)),
}
policy_cfg.output_features = {
    "action": PolicyFeature(type=FeatureType.ACTION, shape=(6,)),
}
policy_cfg.train_expert_only = True
policy_cfg.gradient_checkpointing = True
policy_cfg.dtype = "bfloat16"
policy_cfg.use_amp = True

cfg = TrainPipelineConfig(
    dataset=DatasetConfig(
        repo_id="subhodipsaha/so101_stack_three_cubes_08_19",
        video_backend="pyav",
    ),
    policy=policy_cfg,
    output_dir=Path("/content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2_final"),
    steps=1000,
    batch_size=16,
    num_workers=0,
    log_freq=100,
    save_freq=1000,
)
train(cfg)

Already patched


INFO 2026-08-22 03:18:55 ot_train.py:272 {'batch_size': 16,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataloader_multiprocessing_context': 'spawn',
 'dataset': {'depth_output_unit': 'mm',
             'episodes': None,
             'eval_split': 0.0,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

INFO 2026-08-22 03:24:13 ot_train.py:402 Creating optimizer and scheduler
INFO 2026-08-22 03:24:13 hedulers.py:155 Auto-scaling LR scheduler: num_training_steps (1000) < num_decay_steps (30000). Scaling warmup: 1000 → 33, decay: 30000 → 1000 (scale factor: 0.033)
INFO 2026-08-22 03:24:13 ot_train.py:434 Output dir: /content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2_final
INFO 2026-08-22 03:24:13 ot_train.py:441 cfg.steps=1000 (1K)
INFO 2026-08-22 03:24:13 ot_train.py:442 dataset.num_frames=29148 (29K)
INFO 2026-08-22 03:24:13 ot_train.py:443 dataset.num_episodes=39
INFO 2026-08-22 03:24:13 ot_train.py:446 Effective batch size: 16 x 1 = 16
INFO 2026-08-22 03:24:13 ot_train.py:447 num_learnable_params=693422112 (693M)
INFO 2026-08-22 03:24:13 ot_train.py:448 num_total_params=4143404816 (4B)
Training: 100%|██████████| 1000/1000 [1:46:29<00:00,  6.38s/step]INFO 2026-08-22 05:10:42 ot_train.py:641 step:1K smpl:16K ep:21 epch:0.55 loss:0.050 grdn:0.720 lr:2.7e-06 updt_s:6.090 data_s:0.29

In [ ]:
!find / -name "modeling_pi05.py" 2>/dev/null

/usr/local/lib/python3.13/dist-packages/lerobot/policies/pi05/modeling_pi05.py


In [ ]:
import subprocess, sys, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    sys.executable, "-m", "lerobot.scripts.lerobot_train",
    "--policy.type=pi05",
    "--policy.repo_id=subhodipsaha/pi05_stack_three_cubes_08_19_v2",
    "--dataset.repo_id=subhodipsaha/so101_stack_three_cubes_08_19",
    "--dataset.image_transforms.enable=true",
    "--dataset.video_backend=pyav",
    "--policy.gradient_checkpointing=true",
    "--policy.dtype=bfloat16",
    "--batch_size=4",
    "--steps=20000",
    "--log_freq=100",
    "--save_freq=5000",
    "--output_dir=/content/drive/MyDrive/pi05_stack_three_cubes_08_19_v2",
    "--wandb.enable=false",
]

subprocess.run(cmd, check=True)

KeyboardInterrupt: 